In [ ]:
import pandas as pd
import os
import re
import time
import fitz  # PyMuPDF
import getpass
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

print("1. Setting up API & LLM...")
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("\n2. Processing ALL PDFs with HYBRID FORMULA-AWARE CHUNKING...")
pdf_folder = "../data/raw/"
all_chunks = []

def formula_aware_chunker(text, max_chars=1200):
    segments = re.split(r'(\n\s*(?:[A-Za-z0-9]+\s*=\s*.*|\d+\.\d+\s*[A-Za-z/]+)\s*\n)', text)
    chunks = []
    current = ""
    
    for seg in segments:
        if len(seg) > max_chars:
            if current.strip():
                chunks.append(current.strip())
                current = ""
            
            sub_segments = seg.split('\n\n')
            sub_current = ""
            for sub in sub_segments:
                if len(sub_current) + len(sub) < max_chars:
                    sub_current += sub + "\n"
                else:
                    if sub_current.strip(): chunks.append(sub_current.strip())
                    sub_current = sub + "\n"
            if sub_current.strip(): chunks.append(sub_current.strip())
            
        else:
            if len(current) + len(seg) < max_chars:
                current += seg
            else:
                if current.strip(): chunks.append(current.strip())
                current = seg
                
    if current.strip(): chunks.append(current.strip())
    return chunks

for filename in os.listdir(pdf_folder):
    if filename.endswith(".pdf"):
        print(f"-> Reading {filename}...")
        file_path = os.path.join(pdf_folder, filename)
        doc = fitz.open(file_path)
        full_text = ""
        for page_num in range(min(300, len(doc))):
            full_text += doc.load_page(page_num).get_text("text") + "\n\n"
        
        file_chunks = formula_aware_chunker(full_text)
        all_chunks.extend(file_chunks)
        print(f"   Created {len(file_chunks)} Hybrid Formula-Aware chunks.")

print(f"\n3. Building MASSIVE Vector Database (Total Chunks: {len(all_chunks)})...")
persist_dir = "../data/chroma_formula_large"
vector_db = Chroma.from_texts(
    texts=all_chunks, 
    embedding=embedding_model, 
    collection_name="formula_large_db",
    persist_directory=persist_dir
)

print("\n4. Loading FULL Benchmark Dataset...")
csv_path = "../data/benchmark/physics_evaluation_benchmark.csv"
df = pd.read_csv(csv_path)

print("\n5. Running Evaluation Loop (Rate-Limit Protected)...")
prompt_template = PromptTemplate(
    input_variables=["context", "question", "choices"],
    template="""You are an expert physics professor answering a multiple-choice exam.
Read the context, question, and choices carefully.
You MUST provide your final answer wrapped in XML tags like this: <answer>INDEX</answer>.
If the answer is not clearly in the context, you MUST guess the most logical index based on physics principles.
Only output the XML tag, no other text!

CONTEXT:
{context}

QUESTION: 
{question}

CHOICES:
{choices}

Provide your answer:"""
)

def extract_choices(choices_str):
    matches = re.findall(r"'([^']*)'", str(choices_str))
    if len(matches) != 4: matches = re.findall(r'"([^"]*)"', str(choices_str))
    return matches

evaluation_results = []

for index, row in df.iterrows():
    q_text = row['question']
    expected_ans = row['answer']
    choices_list = extract_choices(row['choices'])
    choices_text = "\n".join([f"Index {i}: {choice}" for i, choice in enumerate(choices_list)])

    retrieved_docs = vector_db.similarity_search(q_text, k=2)
    c_text = "\n\n---\n\n".join([doc.page_content for doc in retrieved_docs])
    
    final_prompt = prompt_template.format(context=c_text, question=q_text, choices=choices_text)
    
    response_text = "" 
    for attempt in range(5):
        try:
            response_text = llm.invoke(final_prompt).content
            time.sleep(2)
            break
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e).lower():
                print(f"   [RATE LIMIT] Sleeping for 20s... (Attempt {attempt+1}/5)")
                time.sleep(20)
            else:
                raise e
    
    if response_text == "":
        pred_ans = "0"
    else:
        match = re.search(r"<answer>\s*(\d+)\s*</answer>", response_text, re.IGNORECASE)
        if match:
            pred_ans = match.group(1)
        else:
            digits = re.findall(r"\d", response_text)
            pred_ans = digits[0] if digits else "0"
        
    print(f"Q{index+1}: {q_text[:40]}...")
    print(f"Expected: {expected_ans} | Predicted: {pred_ans}")
    print("-" * 30)
    
    evaluation_results.append({
        "Question": q_text,
        "Choices": str(choices_list),
        "Expected_Answer": expected_ans,
        "Predicted_Answer": pred_ans,
        "Retrieved_Context": c_text
    })

print("\n6. Saving Formula-Aware Large Scale Results...")
results_df = pd.DataFrame(evaluation_results)
output_path = "../data/benchmark/eval_04_formula_large.csv"
results_df.to_csv(output_path, index=False)
print(f"DONE! Results saved to '{output_path}'")

1. Setting up API & LLM...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


2. Processing ALL PDFs with HYBRID FORMULA-AWARE CHUNKING...
-> Reading university-physics-volume-1.pdf...
   Created 335 Hybrid Formula-Aware chunks.
-> Reading university-physics-volume-2.pdf...
   Created 318 Hybrid Formula-Aware chunks.
-> Reading university-physics-volume-3.pdf...
   Created 338 Hybrid Formula-Aware chunks.

3. Building MASSIVE Vector Database (Total Chunks: 991)...

4. Loading FULL Benchmark Dataset...

5. Running Evaluation Loop (Rate-Limit Protected)...
Q1: The quantum efficiency of a photon detec...
Expected: 1 | Predicted: 3
------------------------------
Q2: White light is normally incident on a pu...
Expected: 2 | Predicted: 0
------------------------------
Q3: Which of the following is true about any...
Expected: 2 | Predicted: 2
------------------------------
Q4: The best type of laser with which to do ...
Expected: 0 | Predicted: 0
------------------------------
Q5: Excited states of the helium atom can be...
Expected: 1 | Predicted: 3
-----------------